# Hard QAOA-ISAC Benchmark

This notebook version of `qaoa_isac_benchmark.py` runs the harder benchmark and displays the results in one place.

It keeps `qaoa_isac_notebook.ipynb` untouched. The original notebook remains the end-to-end Qiskit demo; this notebook is the competition-strength benchmark view.

Important framing: the strongest result below is a constraint-preserving valid-subspace QAOA simulation / QAOA-guided sampler. The optional hardware section prepares a full-binary QUBO QAOA circuit for IBM Quantum hardware; that hardware circuit does not yet implement the custom valid-subspace mixer directly.

## 1. Imports and Parameters

The defaults reproduce the strong benchmark case found in the harness:

- `U=3`, `G=6`, `S=5`, `Nt=4`
- full binary problem size: `18` qubits
- `Gamma_min=0.5`
- seed `35`

In [ ]:
from argparse import Namespace
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np

from qaoa_isac_benchmark import run_benchmark, print_summary

BENCHMARK_ARGS = Namespace(
    uavs=3,
    grid_points=6,
    survivors=5,
    antennas=4,
    gamma_min=0.5,
    seed=35,
    reps=1,
    grid_steps=61,
    shots=1024,
    top_k=12,
    output="qaoa_isac_benchmark_results.json",
    verbose=False,
)

## 2. Run the Benchmark

This executes exact enumeration over feasible assignments, greedy, greedy plus local search, uniform feasible statistics, and the valid-subspace QAOA sampler.

In [ ]:
results = run_benchmark(BENCHMARK_ARGS)
Path(BENCHMARK_ARGS.output).write_text(json.dumps(results, indent=2), encoding="utf-8")
print_summary(results)

## 3. Main Result Table

In [ ]:
def mbps(value):
    return value / 1e6

rows = [
    {
        "method": "Exact enumeration",
        "sum_rate_mbps": mbps(results["exact"]["sum_rate"]),
        "AR_rate": results["exact"]["AR_rate"],
        "assignment": results["exact"]["assignment"],
        "feasible": results["exact"]["feasible"],
    },
    {
        "method": "Greedy",
        "sum_rate_mbps": mbps(results["greedy"]["sum_rate"]),
        "AR_rate": results["greedy"]["AR_rate"],
        "assignment": results["greedy"]["assignment"],
        "feasible": results["greedy"]["feasible"],
    },
    {
        "method": "Greedy + local search",
        "sum_rate_mbps": mbps(results["greedy_local_search"]["sum_rate"]),
        "AR_rate": results["greedy_local_search"]["AR_rate"],
        "assignment": results["greedy_local_search"]["assignment"],
        "feasible": results["greedy_local_search"]["feasible"],
    },
    {
        "method": "Random feasible mean",
        "sum_rate_mbps": mbps(results["random_feasible"]["mean_rate"]),
        "AR_rate": results["random_feasible"]["mean_AR_rate"],
        "assignment": "-",
        "feasible": True,
    },
    {
        "method": "Valid-subspace QAOA top",
        "sum_rate_mbps": mbps(results["valid_subspace_qaoa"]["top_assignment"]["sum_rate"]),
        "AR_rate": results["valid_subspace_qaoa"]["top_assignment"]["AR_rate"],
        "assignment": results["valid_subspace_qaoa"]["top_assignment"]["assignment"],
        "feasible": results["valid_subspace_qaoa"]["top_assignment"]["feasible"],
    },
]

try:
    import pandas as pd
    result_table = pd.DataFrame(rows)
    display(result_table)
except Exception:
    for row in rows:
        print(row)

## 4. QAOA Probability Distribution

The useful win signal is not just that the optimum exists in the feasible set. The valid-subspace QAOA distribution enriches the optimum relative to uniform feasible sampling.

In [ ]:
qaoa = results["valid_subspace_qaoa"]
random = results["random_feasible"]

print(f"Optimum probability under QAOA: {qaoa['optimum_probability']:.6f}")
print(f"Uniform feasible optimum probability: {random['uniform_optimum_probability']:.6f}")
print(f"Enrichment vs uniform: {qaoa['enrichment_vs_uniform']:.2f}x")
print(f"Expected QAOA samples to optimum: {qaoa['expected_samples_to_optimum']:.2f}")
print(f"Expected uniform samples to optimum: {random['expected_samples_to_optimum']:.2f}")
print(f"Shots for 95% optimum-hit probability: {qaoa['shots_for_95pct_success']} QAOA vs {random['shots_for_95pct_success']} uniform")

prob_rows = qaoa["top_probabilities"]
try:
    import pandas as pd
    display(pd.DataFrame(prob_rows))
except Exception:
    print(json.dumps(prob_rows, indent=2))

## 5. Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

labels = [row["method"] for row in rows]
rates = [row["sum_rate_mbps"] for row in rows]
colors = ["#2ca02c", "#ff7f0e", "#8c564b", "#9467bd", "#1f77b4"]
axes[0].bar(labels, rates, color=colors, alpha=0.88)
axes[0].set_ylabel("Sum-rate (Mbps)")
axes[0].set_title("Hard benchmark rate comparison")
axes[0].tick_params(axis="x", rotation=30)
for i, value in enumerate(rates):
    axes[0].text(i, value + 0.08, f"{value:.3f}", ha="center", fontsize=9)

prob_labels = [str(row["assignment"]) for row in prob_rows]
probabilities = [row["probability"] for row in prob_rows]
axes[1].bar(prob_labels, probabilities, color="#1f77b4", alpha=0.88)
axes[1].axhline(random["uniform_optimum_probability"], color="black", linestyle="--", linewidth=1, label="Uniform feasible")
axes[1].set_ylabel("Probability")
axes[1].set_title("Top valid-subspace QAOA probabilities")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Full Results JSON

In [ ]:
print(json.dumps(results, indent=2))

## 7. Optional IBM Quantum Hardware Run

These cells prepare code for a real IBM Quantum run. They are guarded so the notebook can be run safely without submitting a paid or queued hardware job.

What this hardware section does:

- Builds a full-binary QUBO QAOA circuit for the same `U=3, G=6, S=5` benchmark (`18` qubits).
- Uses the QUBO/Ising coefficients from `qaoa_isac_env.py`.
- Uses IBM Runtime `SamplerV2` with backend mode after ISA transpilation.
- Projects measured bitstrings back to feasible assignments for post-processing.

What it does not yet do:

- It does not compile the custom valid-subspace mixer used by the benchmark simulator. That would require a deeper custom circuit synthesis pass for the assignment graph mixer.

Before using real hardware, install Runtime support in the active environment and save your IBM Quantum credentials. IBM's current docs recommend `qiskit-ibm-runtime` for hardware jobs and show `QiskitRuntimeService()`, `service.least_busy(...)`, ISA transpilation with `generate_preset_pass_manager`, and `SamplerV2(mode=backend).run(..., shots=...)`.

In [ ]:
# Hardware execution is opt-in. Leave this False for normal notebook runs.
SUBMIT_TO_HARDWARE = False

# Install if needed in the qiskit environment, then restart the kernel:
#   C:\Users\harry\.conda\envs\qiskit\python.exe -m pip install qiskit-ibm-runtime

# Save credentials once in a trusted local environment, replacing placeholders:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(token="<your-api-key>", instance="<CRN optional>", overwrite=True)

IBM_INSTANCE = None  # set to a CRN string if your account requires a specific instance
HARDWARE_SHOTS = 1024
HARDWARE_REPS = 1

# Starter angles. These are from the valid-subspace benchmark and are not claimed
# to be optimized for the full-binary hardware QUBO circuit.
GAMMA_HW = results["valid_subspace_qaoa"]["gamma"]
BETA_HW = results["valid_subspace_qaoa"]["beta"]

In [ ]:
from collections import Counter

from qiskit import QuantumCircuit

from qaoa_isac_env import ISACEnvironment, SystemParams
from qaoa_isac_benchmark import build_environment, enumerate_assignments

hardware_params = SystemParams(
    U=results["scenario"]["U"],
    G=results["scenario"]["G"],
    S=results["scenario"]["S"],
    Nt=results["scenario"]["Nt"],
    Gamma_min=results["scenario"]["Gamma_min"],
)
hardware_env = build_environment(hardware_params, seed=results["scenario"]["seed"], quiet=True)
feasible_states = enumerate_assignments(hardware_env, require_c3=True)

def scaled_ising_terms(env):
    h = np.array(env.h_bias, dtype=float)
    j = np.array(env.J, dtype=float)
    scale = max(float(np.max(np.abs(h))), float(np.max(np.abs(j))), 1.0)
    return h / scale, j / scale, scale


def build_full_binary_qaoa_circuit(env, gamma, beta, reps=1, measure=True):
    # Build a hardware-executable p-layer QAOA circuit for the full binary QUBO.
    n_qubits = env.n_qubits
    circuit = QuantumCircuit(n_qubits, n_qubits if measure else 0)
    h, j, scale = scaled_ising_terms(env)

    circuit.h(range(n_qubits))
    for _ in range(reps):
        for i, coeff in enumerate(h):
            if abs(coeff) > 1e-12:
                circuit.rz(2.0 * gamma * coeff, i)
        for i in range(n_qubits):
            for k in range(i + 1, n_qubits):
                coeff = j[i, k]
                if abs(coeff) <= 1e-12:
                    continue
                circuit.cx(i, k)
                circuit.rz(2.0 * gamma * coeff, k)
                circuit.cx(i, k)
        for i in range(n_qubits):
            circuit.rx(2.0 * beta, i)

    if measure:
        circuit.measure(range(n_qubits), range(n_qubits))
    circuit.metadata = {
        "problem": "QAOA-ISAC full-binary QUBO",
        "U": env.params.U,
        "G": env.params.G,
        "S": env.params.S,
        "Gamma_min": env.params.Gamma_min,
        "qubo_scale": scale,
        "gamma": gamma,
        "beta": beta,
        "reps": reps,
    }
    return circuit


full_binary_circuit = build_full_binary_qaoa_circuit(
    hardware_env,
    gamma=GAMMA_HW,
    beta=BETA_HW,
    reps=HARDWARE_REPS,
    measure=True,
)

print(f"Hardware candidate circuit: {full_binary_circuit.num_qubits} qubits")
print(f"Depth before ISA transpilation: {full_binary_circuit.depth()}")
print(f"Ops before ISA transpilation: {full_binary_circuit.count_ops()}")

In [ ]:
def bitstring_to_variable_vector(bitstring, n_qubits):
    # Qiskit count strings are displayed with the highest classical bit first.
    return np.array([int(bit) for bit in bitstring[::-1][:n_qubits]], dtype=int)


def assignment_vector(row, env):
    x = np.zeros(env.n_qubits, dtype=int)
    for u, g in enumerate(row.assignment):
        x[u * env.params.G + g] = 1
    return x


def project_bitstring_to_feasible_assignment(bitstring, feasible_rows, env):
    bits = bitstring_to_variable_vector(bitstring, env.n_qubits)
    best_row = None
    best_key = None
    for row in feasible_rows:
        candidate = assignment_vector(row, env)
        hamming = int(np.sum(bits != candidate))
        key = (-hamming, row.sum_rate)
        if best_key is None or key > best_key:
            best_key = key
            best_row = row
    return best_row


def summarize_hardware_counts(counts, feasible_rows, env, exact_rate):
    repaired = Counter()
    for bitstring, count in counts.items():
        row = project_bitstring_to_feasible_assignment(bitstring, feasible_rows, env)
        repaired[row.assignment] += count

    summary = []
    lookup = {row.assignment: row for row in feasible_rows}
    for assignment, count in repaired.most_common(10):
        row = lookup[assignment]
        summary.append({
            "assignment": list(assignment),
            "count": int(count),
            "probability": count / sum(counts.values()),
            "sum_rate_mbps": row.sum_rate / 1e6,
            "AR_rate": row.sum_rate / exact_rate,
        })
    return summary

print("Post-processing helpers are ready.")

In [ ]:
if SUBMIT_TO_HARDWARE:
    from qiskit.transpiler import generate_preset_pass_manager
    from qiskit_ibm_runtime import QiskitRuntimeService
    from qiskit_ibm_runtime import SamplerV2 as Sampler

    service = QiskitRuntimeService(instance=IBM_INSTANCE) if IBM_INSTANCE else QiskitRuntimeService()
    backend = service.least_busy(
        simulator=False,
        operational=True,
        min_num_qubits=full_binary_circuit.num_qubits,
    )
    print(f"Selected backend: {backend.name}")

    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    isa_circuit = pm.run(full_binary_circuit)
    print(f"Depth after ISA transpilation: {isa_circuit.depth()}")
    print(f"Ops after ISA transpilation: {isa_circuit.count_ops()}")

    sampler = Sampler(mode=backend)
    job = sampler.run([(isa_circuit,)], shots=HARDWARE_SHOTS)
    print(f"Job ID: {job.job_id()}")
    print(f"Initial status: {job.status()}")
else:
    print("SUBMIT_TO_HARDWARE is False. Set it to True only when you are ready to submit a real IBM Quantum job.")

In [ ]:
# Run this cell after a real hardware job completes.
# It assumes a variable named `job` exists from the submission cell.
if "job" in globals():
    hardware_result = job.result()
    hardware_counts = hardware_result[0].data.meas.get_counts()
    hardware_summary = summarize_hardware_counts(
        hardware_counts,
        feasible_states,
        hardware_env,
        exact_rate=results["exact"]["sum_rate"],
    )
    try:
        import pandas as pd
        display(pd.DataFrame(hardware_summary))
    except Exception:
        print(json.dumps(hardware_summary, indent=2))
else:
    print("No hardware job is available yet. Submit a job first, or retrieve one by job ID with QiskitRuntimeService().job('<job-id>').")